In [1]:
"""
Created on June 6, 2026
@author: Carl Emil Elling
Calculating SNR of stitched PL image
"""

import os
import cv2
import numpy as np
from tkinter import filedialog
import tifffile
from src.utils import ingaas_processing
from src import stitching
from src.stitching import helpers
import matplotlib.pyplot as plt

def calculate_snr(image):
    # Preliminary whole-image SNR estimate in dB.
    image_array = np.asarray(image, dtype=np.float64)
    if image_array.size == 0:
        raise ValueError("Cannot calculate SNR for an empty image.")

    signal_mean = float(np.mean(image_array))
    noise_std = float(np.std(image_array))

    if noise_std == 0:
        return float("inf") if signal_mean > 0 else 0.0

    snr_linear = abs(signal_mean) / noise_std
    return 20.0 * np.log10(snr_linear)

def load_images(source, width, height):
    if source.endswith('.raw'):
        print("Processing a .raw file")
        images = ingaas_processing.load_raw_image(source, width, height)
    elif source.endswith('.tiff'):
        print("Processing a .tiff file") 
        images = tifffile.imread(source)
    else:
        raise ValueError("Unsupported file format. Please provide a .tiff or .raw file.")
    return images



In [ ]:

# Prompt user to select an image file
im_path = filedialog.askopenfilename(title='Select an image to process')

# Read the image using tifffile
width, height = 640, 512  # Example dimensions, adjust as necessary
images = load_images(im_path, width, height)  # Adjust width and height as needed

# Calculate SNR
#snr_value = calculate_snr(images)

#print(f"SNR of the image: {snr_value:.2f} dB")


In [ ]:
#Functions from Jeppe
def get_phase(x, f, f_s, debug_out=False):
    '''
    Finds the phase of a given frequency in a signal, sampled at some other frequency.

    Args:
        x: The signal as a numpy array
        f: The frequency to find the phase of
        f_s: The frequency at which the signal was sampled
        debug_out: Whether to display some debug info

    Returns:
        phase: The phase of the signal
        f_out: The closest frequency present in the FFT

    '''

    # Where is the modulation frequency in the results?
    fft_frequencies = scipy.fft.fftshift(scipy.fft.rfftfreq(len(x), d=1/f_s))
    f_index = np.argmin(np.abs(f-fft_frequencies))
    f_out = fft_frequencies[f_index]
    if debug_out:
        print(f_index)
        print(f_out)

    # FFT of data
    yf = scipy.fft.fftshift(scipy.fft.rfft(x))
    if debug_out:
        plt.plot(fft_frequencies, yf)
        plt.show()


    # Get phase
    phase = np.angle(yf[f_index])
    if debug_out:
        print(yf[f_index])
        print("Phase:", phase)

    return phase, f_out

def get_extrema(f,phase,f_s,N):
    '''
    Estimate the extrema points of a discrete signal

    Args:
        f: Frequency of the signal component of interest
        phase: How the signal component is shifted. Should be in the range [-Pi,Pi]
        f_s: Frequency at which the signal is sampled
        start: Index at which to start return from
        end: Index at which to end return at

    Returns:
        peaks: Index of peaks as a numpy array
        valleys: Index of valleys as a numpy array

    '''
    peak_arg =  -phase

    if peak_arg < 0:
        peak_arg += 2*np.pi

    if peak_arg > 2*np.pi:
        peak_arg -= 2*np.pi

    valley_arg = peak_arg + np.pi

    base = np.arange(N)*2*np.pi
    peaks = np.round(f_s*(base+peak_arg)/(2*np.pi*f)).astype(np.int32)
    valleys = np.round(f_s*(base+valley_arg)/(2*np.pi*f)).astype(np.int32)

    #mask = np.all([peaks > start, peaks < end, valleys > start, valleys < end], axis=0)
    mask = np.all([peaks < N, valleys < N], axis=0)
    return peaks[mask], valleys[mask]